# MELG607 Full-Period Search and Equidistribution

Search for full-period MELG generators with $k = 607$ ($2^{607}-1$ period),
then test each with 100 random Lagged Tempering parameter sets.
Generators with dimension gaps sum less than 10 are saved to
`yaml/testedgenerators/`.

In [ ]:
import random
import time
import yaml

from regpoly.search_primitive import PrimitiveSearch
from regpoly.generateur import Generateur
from regpoly.transformation import Transformation
from regpoly.combinaison import Combinaison
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest, METHOD_MATRICIAL
)
from regpoly.tested_generator import save_tested_generator

## 1. Search for full-period MELG607 generators

MELG parameters: `w=64`, `N=10` (9 array words + lung), `r=33`.
Structural: `w`, `N`, `r`.  Randomized: `M`, `sigma1`, `sigma2`, `a`.

$k = N \times w - r = 10 \times 64 - 33 = 607$

In [ ]:
search = PrimitiveSearch(
    family="MELG",
    L=64,
    structural={"w": 64, "N": 10, "r": 33},
    search_params={},
    max_tries=50000,
    max_seconds=None,
    generators_dir="../../yaml/generators",
)

print(f"Output file: {search.output_file}")
results = search.run()

## 2. Load found generators

In [ ]:
with open(search.output_file) as f:
    data = yaml.safe_load(f)

family = data["family"]
common = data["common"]
n_gens = len(data["generators"])
print(f"Loaded {n_gens} full-period MELG607 generators")
print(f"Structural: {common}")
print()
for i, entry in enumerate(data["generators"][:5], 1):
    print(f"  {i}. M={entry['M']}, sigma1={entry['sigma1']}, "
          f"sigma2={entry['sigma2']}, a=0x{entry['a']:016x}")
if n_gens > 5:
    print(f"  ...and {n_gens - 5} more")

## 3. Equidistribution with Lagged Tempering

For each full-period generator, try 100 random Lagged Tempering
parameter sets (random `sigma`, `L`, and `b`).  Save those with
dimension gaps sum less than 10.

In [ ]:
INT_MAX = 2**31 - 1
SE_THRESHOLD = 10
NBTRIES = 100
w = common["w"]
arr_size = common["N"] - 1  # number of array words

print(f"Testing {n_gens} generators x {NBTRIES} tempering tries "
      f"= {n_gens * NBTRIES} tests")
print(f"Threshold: se < {SE_THRESHOLD}")
print()

good = []
t0 = time.time()

for i, entry in enumerate(data["generators"]):
    gen = Generateur.create(family, w, **common, **entry)

    for t in range(NBTRIES):
        sigma = random.randint(1, w - 1)
        L = random.randint(1, arr_size - 2)
        b = random.getrandbits(w)

        trans = Transformation("laggedTempering",
                               {"w": w, "sigma": sigma, "L": L, "b": b}, w)

        comb = Combinaison(J=1, Lmax=w)
        comb.components[0].add_gen(gen)
        comb.components[0].add_trans(trans)
        comb.reset()

        test = EquidistributionTest(L=w, delta=[INT_MAX] * (w + 1),
                                    mse=INT_MAX, method=METHOD_MATRICIAL)
        result = test.run(comb)

        if result.se < SE_THRESHOLD:
            test_results = {"equidistribution": {"se": result.se}}
            ecart = {l: result.ecart[l]
                     for l in range(1, w + 1) if result.ecart[l] != 0}
            if ecart:
                test_results["equidistribution"]["ecart"] = ecart
            if result.is_me():
                test_results["equidistribution"]["status"] = "ME"

            path = save_tested_generator(
                "../../yaml/testedgenerators", "equidist", comb, test_results)

            good.append({
                "gen": entry,
                "sigma": sigma, "L": L, "b": b,
                "se": result.se,
                "is_me": result.is_me(),
                "path": path,
            })
            print(f"  gen {i+1:>3d} try {t+1:>3d}: "
                  f"M={entry['M']}, a=0x{entry['a']:016x}, "
                  f"sigma={sigma}, L={L}, b=0x{b:016x} "
                  f"-> se={result.se}"
                  f"{'  ME!' if result.is_me() else ''}")

elapsed = time.time() - t0
print(f"\nFound {len(good)} generators with se < {SE_THRESHOLD} "
      f"({elapsed:.1f}s)")

## 4. Results sorted by dimension gaps sum

In [ ]:
if good:
    ranked = sorted(good, key=lambda x: x["se"])
    print(f"{'#':<3} {'M':<3} {'s1':<3} {'s2':<3} {'a':<18} "
          f"{'sig':<4} {'L':<3} {'b':<18} {'se':>3} {'ME':>3}")
    print("-" * 75)
    for rank, g in enumerate(ranked, 1):
        e = g["gen"]
        print(f"{rank:<3} {e['M']:<3} {e['sigma1']:<3} {e['sigma2']:<3} "
              f"0x{e['a']:016x} "
              f"{g['sigma']:<4} {g['L']:<3} 0x{g['b']:016x} "
              f"{g['se']:>3} {'yes' if g['is_me'] else '':>3}")
    print(f"\nTotal: {len(good)} generators saved to yaml/testedgenerators/MELG/")
else:
    print(f"No generators found with se < {SE_THRESHOLD}.")

## 5. Equidistribution table for the best generator

In [ ]:
from regpoly.tested_generator import load_tested_generator

if good:
    best = sorted(good, key=lambda x: x["se"])[0]
    comb, saved_results = load_tested_generator(best["path"])

    e = best["gen"]
    print(f"Best generator:")
    print(f"  M={e['M']}, sigma1={e['sigma1']}, sigma2={e['sigma2']}, "
          f"a=0x{e['a']:016x}")
    print(f"  Tempering: sigma={best['sigma']}, L={best['L']}, "
          f"b=0x{best['b']:016x}")
    print(f"  Saved: {best['path']}")
    print()

    test = EquidistributionTest(L=comb.L, delta=[INT_MAX] * (comb.L + 1),
                                mse=INT_MAX, method=METHOD_MATRICIAL)
    result = test.run(comb)

    print(f"Dimension gaps sum = {result.se}")
    if result.is_me():
        print("==> ME generator!")
    print()

    table_str, _ = result.display_table(comb, 'l')
    print(table_str)
else:
    print(f"No generators found with se < {SE_THRESHOLD}.")